# Lab 3: Integer and Binary Programming

In the previous labs, variables could often take fractional values. In many automation problems that is not realistic: we either buy a robot arm or we do not, a person either works a shift or does not, and one task either happens before another task or after it.

This lab focuses on **binary variables**. A binary variable is a switch:

$$
x_i =
\begin{cases}
1 & \text{if option } i \text{ is selected} \\
0 & \text{otherwise}
\end{cases}
$$

By the end of the lab you should be able to use binary variables to model:

- yes/no choices,
- logical rules such as "if A then B",
- fixed startup costs,
- realistic shift schedules.


In [ ]:
# This cell keeps the notebook runnable in Google Colab.
# For local work, install dependencies first with:
#   python -m pip install -r requirements.txt
import importlib.util
import subprocess
import sys

required_packages = ["pulp", "pandas", "matplotlib"]
missing_packages = [
    package for package in required_packages
    if importlib.util.find_spec(package) is None
]

if missing_packages:
    try:
        import google.colab  # noqa: F401
        running_in_colab = True
    except ImportError:
        running_in_colab = False

    if running_in_colab:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing_packages,
        ])
    else:
        missing_text = ", ".join(missing_packages)
        raise ImportError(
            f"Missing packages: {missing_text}. "
            "For local setup, run: python -m pip install -r requirements.txt"
        )

# Select the plotting backend before importing pyplot.
# In Colab/Jupyter this keeps plots inside the notebook output cell.
# In a plain Python script this prevents separate GUI windows.
try:
    ipython = get_ipython()
except NameError:
    ipython = None

if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")
else:
    import matplotlib
    matplotlib.use("Agg")

import pulp
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


## Binary modeling patterns

The important idea is not only that a variable is 0 or 1. The important idea is that binary variables let us put **logic** inside an optimization model.

| Modeling idea | Typical linear constraint |
| --- | --- |
| Choose at most one option | $\sum_i x_i \le 1$ |
| Choose exactly one option | $\sum_i x_i = 1$ |
| If A is chosen, B must also be chosen | $x_A \le x_B$ |
| A and B cannot both happen | $x_A + x_B \le 1$ |
| Activity quantity only if switched on | $q \le U y$ |
| If switched on, do at least $L$ units | $q \ge L y$ |
| Fixed startup cost | objective contains $F y + c q$, with $q \le U y$ |
| Either constraint 1 or constraint 2 | use Big-M and a binary switch |

Read the examples below as modeling templates. The numbers are small so that you can inspect the solution and ask whether it makes sense.


## Example 1: choosing robot-lab upgrades

A robotics lab has a limited budget before an open day. It can buy or upgrade several pieces of equipment. Each project has a cost and an educational/demo value.

Some choices depend on others:

- the vision camera is useful only if the robot arm is selected,
- the conveyor requires the safety cage,
- the lab does not have enough supervision to use both the robot arm and the mobile base during the open day,
- at least one automation demo must be possible, so either the vision camera or conveyor must be selected.

The binary decision variable is:

$$x_p = 1 \quad \text{if project } p \text{ is selected.}$$


In [ ]:
def print_solution(model):
    """Solve a PuLP model and print variables with nonzero values first."""
    status = model.solve(pulp.PULP_CBC_CMD(msg=False))
    print("Status:", pulp.LpStatus[status])

    variables = sorted(model.variables(), key=lambda variable: variable.name)
    nonzero = [variable for variable in variables if abs(pulp.value(variable) or 0) > 1e-7]
    zero = [variable for variable in variables if abs(pulp.value(variable) or 0) <= 1e-7]

    for variable in nonzero + zero:
        value = pulp.value(variable)
        if value is None:
            print(f"{variable.name} = None")
        elif abs(value - round(value)) < 1e-7:
            print(f"{variable.name} = {int(round(value))}")
        else:
            print(f"{variable.name} = {value:.3f}")

    if model.objective is not None:
        print("Objective value =", pulp.value(model.objective))


In [ ]:
projects = {
    "robot_arm": {"cost": 6, "value": 9},
    "vision_camera": {"cost": 4, "value": 7},
    "mobile_base": {"cost": 5, "value": 6},
    "safety_cage": {"cost": 3, "value": 2},
    "conveyor": {"cost": 4, "value": 5},
}

budget = 13

model = pulp.LpProblem("Robot_Lab_Investment", pulp.LpMaximize)

x = {
    name: pulp.LpVariable(name, cat="Binary")
    for name in projects
}

# Objective: maximize educational/demo value.
model += pulp.lpSum(projects[p]["value"] * x[p] for p in projects), "total_demo_value"

# Budget.
model += pulp.lpSum(projects[p]["cost"] * x[p] for p in projects) <= budget, "budget"

# Logic constraints.
model += x["vision_camera"] <= x["robot_arm"], "camera_requires_robot_arm"
model += x["conveyor"] <= x["safety_cage"], "conveyor_requires_safety_cage"
model += x["robot_arm"] + x["mobile_base"] <= 1, "not_enough_supervision_for_both"
model += x["vision_camera"] + x["conveyor"] >= 1, "at_least_one_automation_demo"

print_solution(model)


In [ ]:
selection_rows = []
for project, data in projects.items():
    selected = int(round(pulp.value(x[project])))
    selection_rows.append({
        "project": project,
        "selected": selected,
        "cost": data["cost"],
        "value": data["value"],
        "value_per_cost": round(data["value"] / data["cost"], 2),
    })

selection_table = pd.DataFrame(selection_rows)
display(selection_table)
print("Total cost:", sum(row["selected"] * row["cost"] for row in selection_rows))
print("Budget:", budget)


### Check your understanding

Answer these before changing the model:

1. Which selected project has the highest value?
2. Which selected project has the highest value per cost?
3. Does the optimizer simply choose the best value-per-cost projects? Explain using one constraint from the model.
4. Which logical rule is written as `x["vision_camera"] <= x["robot_arm"]`?
5. Which rule is written as `x["robot_arm"] + x["mobile_base"] <= 1`?

### Task 1: budget sensitivity

Run the model for budgets from 8 to 18. For each budget, record the selected projects and total value.

Use this four-step structure in your answer:

1. **Prediction:** At what budget do you expect the solution to change?
2. **Model change:** Which parameter did you change?
3. **Result:** At which budgets did the solution change?
4. **Interpretation:** Which logical constraints explain those changes?


In [ ]:
# Task 1 workspace.
# Hint: turn the model above into a function solve_robot_lab(budget),
# then call it inside a loop for budgets from 8 to 18.


## Example 2: either-or constraints with Big-M

Sometimes one of two constraints must hold, but not necessarily both.

Example:

Maximize $x + y$ where $0 \le x \le 10$, $0 \le y \le 10$, and either

$$x + y \le 3$$

or

$$3x - y \le 3$$

must be satisfied.

A binary variable can choose which constraint is relaxed. This is called a **Big-M** formulation. Big-M should be large enough to relax a constraint when needed, but not absurdly large.

Here, $x$ and $y$ can never exceed 10, so $M = 20$ is enough.


In [ ]:
logic_model = pulp.LpProblem("Alternative_Constraints_Problem", pulp.LpMaximize)

x_logic = pulp.LpVariable("x", lowBound=0, upBound=10, cat="Continuous")
y_logic = pulp.LpVariable("y", lowBound=0, upBound=10, cat="Continuous")

# z = 0 relaxes the second constraint.
# z = 1 relaxes the first constraint.
z = pulp.LpVariable("relax_first_constraint", cat="Binary")

logic_model += x_logic + y_logic, "simple_sum"

M = 20
logic_model += x_logic + y_logic <= 3 + M * z, "first_constraint"
logic_model += 3 * x_logic - y_logic <= 3 + M * (1 - z), "second_constraint"

print_solution(logic_model)
print("\nConstraint check:")
print("x + y =", pulp.value(x_logic + y_logic), "must be <= 3 if z = 0")
print("3x - y =", pulp.value(3 * x_logic - y_logic), "must be <= 3 if z = 1")


### Check your understanding

1. What value did the solver choose for `relax_first_constraint`?
2. Which of the two original constraints is actually enforced in the solution?
3. What happens if you set `M = 2`? Is the model still representing the intended either-or rule?
4. Why is `M = 20` better than `M = 10000` here?


In [ ]:
# Big-M exploration workspace.
# Try changing M above to 2, 10, 20, and 10000.
# Record the solution and explain whether the model still means the same thing.


## Example 3: startup costs and binary activation

Binary variables are also useful when a cost changes shape.

A lab can assemble demo kits manually or with a robot:

- manual assembly costs 8 per kit,
- robot assembly costs 3 per kit,
- switching on and calibrating the robot costs 30,
- the robot can make at most 20 kits.

The robot cost is really:

$$
\text{robot cost} =
\begin{cases}
0 & \text{if the robot is not used} \\
30 + 3q & \text{if the robot produces } q > 0
\end{cases}
$$

That is not linear as written. We make it linear with a binary switch `use_robot` and the constraint:

$$q \le 20 \cdot \text{use\_robot}.$$

If `use_robot = 0`, robot production must be 0. If `use_robot = 1`, robot production can be as high as 20.


In [ ]:
def solve_startup_cost(demand, fixed_cost=30, minimum_robot_batch=0):
    model = pulp.LpProblem("Startup_Cost_Demo", pulp.LpMinimize)

    manual = pulp.LpVariable("manual_kits", lowBound=0, cat="Integer")
    robot = pulp.LpVariable("robot_kits", lowBound=0, cat="Integer")
    use_robot = pulp.LpVariable("use_robot", cat="Binary")

    robot_capacity = 20

    # Objective: variable costs plus robot startup cost.
    model += 8 * manual + 3 * robot + fixed_cost * use_robot, "total_cost"

    # Produce enough kits.
    model += manual + robot >= demand, "meet_demand"

    # Activation logic.
    model += robot <= robot_capacity * use_robot, "robot_only_if_switched_on"

    # Optional minimum batch size.
    if minimum_robot_batch > 0:
        model += robot >= minimum_robot_batch * use_robot, "minimum_robot_batch"

    status = model.solve(pulp.PULP_CBC_CMD(msg=False))

    return {
        "demand": demand,
        "status": pulp.LpStatus[status],
        "manual": int(round(pulp.value(manual))),
        "robot": int(round(pulp.value(robot))),
        "use_robot": int(round(pulp.value(use_robot))),
        "cost": pulp.value(model.objective),
    }

startup_results = pd.DataFrame([
    solve_startup_cost(demand)
    for demand in range(1, 16)
])

display(startup_results)


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(startup_results["demand"], startup_results["cost"], marker="o")
plt.xlabel("Demand, kits")
plt.ylabel("Minimum cost")
plt.title("When does the robot become worth switching on?")
plt.grid(True, alpha=0.3)
plt.show()


### Task 2: break-even exploration

Use the table and plot above.

1. **Prediction:** If the startup cost increases from 30 to 50, will the robot switch on earlier or later?
2. **Model change:** Change `fixed_cost` in `solve_startup_cost`.
3. **Result:** For which demand is `use_robot = 1` for the first time?
4. **Interpretation:** Why does a cheaper per-kit robot still stay off for small demand?

Then add a minimum batch size:

```python
solve_startup_cost(demand=10, fixed_cost=30, minimum_robot_batch=4)
```

Explain what the constraint `robot >= 4 * use_robot` means in words.


In [ ]:
# Task 2 workspace.
# Try fixed_cost=50 and minimum_robot_batch=4.


## Example 4: connected shift scheduling

The robotics lab is open for student demos. Some hours are busier than others. During peak hours, more assistants are needed.

A simple hour-by-hour model can create unrealistic schedules, for example a person works 8-9, then 13-14, then 17-18. In a real lab, if someone comes in, they usually work one connected block.

So instead of deciding hour by hour, we generate all possible **connected shift blocks** and choose among them.

The binary decision variable is:

$$\text{choose\_shift}_i = 1 \quad \text{if shift block } i \text{ is selected.}$$


In [ ]:
days = ["Mon", "Tue", "Wed", "Thu"]
hours = list(range(8, 18))  # 8 means 8-9, 17 means 17-18.

people = {
    "Anna": {
        "rate": 17,
        "max_week": 12,
        "availability": {
            "Mon": [(8, 14)],
            "Tue": [(8, 16)],
            "Wed": [(8, 12)],
            "Thu": [(10, 16)],
        },
    },
    "Ben": {
        "rate": 15,
        "max_week": 14,
        "availability": {
            "Mon": [(10, 16)],
            "Tue": [(12, 18)],
            "Wed": [(8, 16)],
            "Thu": [(8, 14)],
        },
    },
    "Cara": {
        "rate": 16,
        "max_week": 12,
        "availability": {
            "Mon": [(8, 12)],
            "Tue": [(8, 14)],
            "Wed": [(12, 18)],
            "Thu": [(8, 16)],
        },
    },
    "Mark": {
        "rate": 12,
        "max_week": 16,
        "availability": {
            "Mon": [(8, 18)],
            "Tue": [(8, 18)],
            "Wed": [(8, 18)],
            "Thu": [(8, 18)],
        },
    },
}

# Base demand: one person per hour.
required = {
    (day, hour): 1
    for day in days
    for hour in hours
}

# Peak hours: two people needed on Tuesday and Wednesday from 11-15.
for hour in range(11, 15):
    required[("Tue", hour)] = 2
    required[("Wed", hour)] = 2

min_shift = 2
max_shift = 5


In [ ]:
def generate_connected_shifts(people, days, min_shift=2, max_shift=5):
    shifts = []

    for person, data in people.items():
        for day in days:
            for available_start, available_end in data["availability"].get(day, []):
                for start in range(available_start, available_end):
                    latest_end = min(available_end, start + max_shift)
                    for end in range(start + min_shift, latest_end + 1):
                        shifts.append({
                            "person": person,
                            "day": day,
                            "start": start,
                            "end": end,
                            "duration": end - start,
                        })

    return shifts

shifts = generate_connected_shifts(people, days, min_shift, max_shift)
shifts_table = pd.DataFrame(shifts)
print("Number of possible connected shifts:", len(shifts_table))
display(shifts_table.head(10))


Each row above is a possible shift block. The optimizer will choose a subset of these rows.

This is the key modeling decision: **connected shifts are built into the options**, so the solver cannot create split shifts.


In [ ]:
def solve_connected_shift_schedule(people, days, hours, required, min_shift=2, max_shift=5):
    shifts = generate_connected_shifts(people, days, min_shift, max_shift)

    model = pulp.LpProblem("Connected_Shift_Scheduling", pulp.LpMinimize)

    choose = {
        i: pulp.LpVariable(f"choose_shift_{i}", cat="Binary")
        for i in range(len(shifts))
    }

    # Minimize wage cost.
    wage_cost = pulp.lpSum(
        people[shift["person"]]["rate"] * shift["duration"] * choose[i]
        for i, shift in enumerate(shifts)
    )
    model += wage_cost, "wage_cost"

    # Coverage constraints.
    for day in days:
        for hour in hours:
            model += (
                pulp.lpSum(
                    choose[i]
                    for i, shift in enumerate(shifts)
                    if shift["day"] == day and shift["start"] <= hour < shift["end"]
                )
                >= required[(day, hour)]
            ), f"coverage_{day}_{hour}"

    # Each person can work at most one connected block per day.
    for person in people:
        for day in days:
            model += (
                pulp.lpSum(
                    choose[i]
                    for i, shift in enumerate(shifts)
                    if shift["person"] == person and shift["day"] == day
                )
                <= 1
            ), f"one_shift_per_day_{person}_{day}"

    # Weekly hour limits.
    for person in people:
        model += (
            pulp.lpSum(
                shift["duration"] * choose[i]
                for i, shift in enumerate(shifts)
                if shift["person"] == person
            )
            <= people[person]["max_week"]
        ), f"weekly_limit_{person}"

    status = model.solve(pulp.PULP_CBC_CMD(msg=False))
    status_name = pulp.LpStatus[status]

    if status_name != "Optimal":
        print(f"No valid schedule found. Solver status: {status_name}")
        return model, pd.DataFrame(columns=[
            "person",
            "day",
            "start",
            "end",
            "duration",
            "cost",
        ])

    selected_shifts = pd.DataFrame([
        shift
        for i, shift in enumerate(shifts)
        if pulp.value(choose[i]) > 0.5
    ])

    if not selected_shifts.empty:
        selected_shifts = selected_shifts.sort_values(["day", "start", "person"]).reset_index(drop=True)
        selected_shifts["cost"] = selected_shifts.apply(
            lambda row: people[row["person"]]["rate"] * row["duration"],
            axis=1,
        )

    return model, selected_shifts

shift_model, selected_shifts = solve_connected_shift_schedule(
    people, days, hours, required, min_shift, max_shift
)

shift_status = pulp.LpStatus[shift_model.status]
print("Status:", shift_status)
if shift_status == "Optimal":
    print("Total wage cost:", pulp.value(shift_model.objective))
    display(selected_shifts)
else:
    print("No selected shifts are shown because the model has no valid optimal schedule.")


In [ ]:
def summarize_coverage(selected_shifts, days, hours, required):
    rows = []
    for day in days:
        for hour in hours:
            if selected_shifts.empty:
                assigned_people = []
            else:
                assigned_people = selected_shifts[
                    (selected_shifts["day"] == day)
                    & (selected_shifts["start"] <= hour)
                    & (hour < selected_shifts["end"])
                ]["person"].tolist()

            rows.append({
                "day": day,
                "hour": f"{hour}:00-{hour + 1}:00",
                "required": required[(day, hour)],
                "assigned": len(assigned_people),
                "people": ", ".join(assigned_people),
            })

    return pd.DataFrame(rows)

coverage_table = summarize_coverage(selected_shifts, days, hours, required)
display(coverage_table)


In [ ]:
def plot_shift_schedule(selected_shifts):
    if selected_shifts.empty:
        print("No shifts selected.")
        return

    people_order = list(selected_shifts["person"].drop_duplicates())
    day_order = list(selected_shifts["day"].drop_duplicates())
    y_labels = [f"{day} - {person}" for day in day_order for person in people_order]
    y_position = {label: index for index, label in enumerate(y_labels)}

    plt.figure(figsize=(10, max(4, 0.35 * len(y_labels))))
    for row in selected_shifts.itertuples(index=False):
        label = f"{row.day} - {row.person}"
        plt.barh(
            y_position[label],
            row.duration,
            left=row.start,
            height=0.6,
            edgecolor="black",
        )

    plt.yticks(range(len(y_labels)), y_labels)
    plt.xticks(range(8, 19))
    plt.xlabel("Time")
    plt.title("Selected connected shifts")
    plt.grid(axis="x", alpha=0.3)
    plt.show()

plot_shift_schedule(selected_shifts)


### Check your understanding

1. Why did we create possible shift blocks before building the optimization model?
2. Which constraint guarantees that each person works at most one block per day?
3. Which hours are peak hours?
4. Does the cheapest person, Mark, work every possible hour? Which constraints prevent that?
5. Compare the coverage table with the selected shifts. Are all required hours covered?

### Task 3: peak-hour experiment

Add a new peak period on Thursday:

```python
for hour in range(12, 16):
    required[("Thu", hour)] = 3
```

Then solve again.

Use this four-step answer:

1. **Prediction:** Will the model still be feasible?
2. **Model change:** Which demand values did you change?
3. **Result:** Which people are used during the new peak?
4. **Interpretation:** Which constraints make the solution more expensive or infeasible?


In [ ]:
# Task 3 workspace.
# Copy required to a new dictionary before changing it:
# required_experiment = required.copy()
# for hour in range(12, 16):
#     required_experiment[("Thu", hour)] = 3
# Then call solve_connected_shift_schedule(..., required_experiment, ...).
# Always check pulp.LpStatus[experiment_model.status] before interpreting shifts.
# If the status is Infeasible, explain which limits make the new demand impossible.


## Optional extensions

Choose one experiment, predict the answer, solve it, and explain the result.

| Difficulty | Experiment |
| --- | --- |
| Easy | Change one person's hourly rate. Does the solution change? |
| Easy | Change maximum shift length from 5 to 4. |
| Medium | Add a new person with limited availability but low cost. |
| Medium | Require at least one robot-trained person during every peak hour. |
| Medium | Add fairness: no person should work more than 4 hours more than another. |
| Hard | Allow uncovered hours, but add a large penalty for each uncovered hour. |
| Hard | Compare split-shift scheduling with connected-shift scheduling. |
| Hard | Add a lunch-break rule for shifts longer than 4 hours. |

Use the same structure every time:

1. **Prediction:** What do you expect to happen?
2. **Model change:** What variable, constraint, or parameter did you add?
3. **Result:** What changed in the solution?
4. **Interpretation:** Why did the optimizer choose that?


## Reflection

In this lab, binary variables played four different roles:

1. **Choices:** select or reject robot-lab upgrades.
2. **Logic:** enforce dependencies, either-or rules, and mutual exclusion.
3. **Activation:** switch on the robot only when its fixed cost is worth paying.
4. **Schedule structure:** choose realistic connected shift blocks.

When you build your own model, start by asking:

- What are the decisions?
- Which decisions are yes/no switches?
- Which rules are logical rather than numerical?
- Which outputs should I inspect to check whether the solution makes sense?
